In [2]:
import cv2
import pandas as pd
from ultralytics import YOLO
from datetime import datetime

# Load YOLO Model
model = YOLO("yolov8n.pt")

# Video Path
video_path = "input.mp4"  
cap = cv2.VideoCapture(video_path)

# Output Video
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

out = cv2.VideoWriter(
    "output.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

# ROI Coordinates
# (x1, y1, x2, y2)
roi = (150, 100, 500, 450)

# Variables
inside_people = set()
entry_time = {}
logs = []

while cap.isOpened():
    ret, frame = cap.read()

    if not ret:
        break

    x1, y1, x2, y2 = roi
    # Draw ROI
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,255), 2)
    cv2.putText(frame, "ROI", (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)

    # Tracking
    results = model.track(frame, persist=True, verbose=False)

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy().astype(int)
        classes = results[0].boxes.cls.cpu().numpy().astype(int)

        current_inside = set()

        for box, track_id, cls in zip(boxes, ids, classes):
            # Person class = 0
            if cls != 0:
                continue

            bx1, by1, bx2, by2 = map(int, box)

            cx = (bx1 + bx2) // 2
            cy = (by1 + by2) // 2

            cv2.rectangle(frame, (bx1, by1), (bx2, by2), (0,255,0), 2)
            cv2.putText(frame, f"ID {track_id}", (bx1, by1-8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

            cv2.circle(frame, (cx, cy), 4, (0,0,255), -1)

            # Check if center is inside ROI
            if x1 < cx < x2 and y1 < cy < y2:
                current_inside.add(track_id)
                if track_id not in inside_people:
                    inside_people.add(track_id)
                    time_now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    entry_time[track_id] = time_now
                    logs.append({
                        "Person ID": track_id,
                        "Event": "Entered",
                        "Time": time_now
                    })

        # Detect Exit
        left_people = inside_people - current_inside

        for pid in left_people:
            time_now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            logs.append({
                "Person ID": pid,
                "Event": "Exited",
                "Time": time_now
            })

            inside_people.remove(pid)

            if pid in entry_time:
                del entry_time[pid]

    # Active Count
    cv2.putText(frame,
                f"People Inside ROI: {len(inside_people)}",
                (20,40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0,0,255),
                2)

    out.write(frame)

    cv2.imshow("Security Monitoring", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
out.release()
cv2.destroyAllWindows()

# Save CSV
df = pd.DataFrame(logs)
df.to_csv("event_logs.csv", index=False)

print("Done!")
print("Processed video saved as: output.mp4")
print("Event log saved as: event_logs.csv")

Done!
Processed video saved as: output.mp4
Event log saved as: event_logs.csv
